# Week 2 · Member 6: Hashing patterns
## Frequency counts and two-sum — a beginner-friendly Python lesson
**Question round:** Wednesday. **Friday build:** returns-desk queue and wait times.

This notebook has exactly **30 cells**. Use Python 3 in Jupyter or Google Colab. No extra packages, internet connection, or API server is needed.

**Source:** [HARBOUR learning plan, learn branch](https://github.com/Snrboi/harbour/blob/learn/README.md), Week 2, Member 6. Examples below are original, fictional teaching examples.  

## 2 · What learners should be able to do
- Count repeated items with a dictionary.
- Explain the time–memory trade-off of hashing.
- Find two distinct list positions whose values add to a target.
- Test empty inputs, repeated values, and missing answers.
- Use a queue to model a returns desk and calculate waiting times.

**Suggested session:** 10 minutes on dictionaries, 15 on counting, 20 on two-sum, 15 on questions/practice. Use the queue section for Friday's build.
**Before starting:** learners should know variables, lists, loops, functions, and `if`. Ask: “How would you count repeated book titles by hand?”

## 3 · A dictionary is a labelled storage box
A list stores items by position. A dictionary stores **key → value** pairs.

For counting, the key is a book title and the value is its count. Hashing helps Python locate a key without checking every entry in sequence. Different keys can have the same hash; Python handles these collisions internally.

Dictionary lookup and update are **O(1) on average**, not guaranteed constant time. In a bad collision case, one operation can be O(n). We use average-case dictionary costs in this lesson.

In [3]:
book_counts = {"Python Basics": 2, "Ocean Stories": 1}
print(book_counts["Python Basics"])  # 2
print(book_counts.get("New Book", 0))  # 0: default for a missing key
print(book_counts)  # get() did not insert "New Book"

2
0
{'Python Basics': 2, 'Ocean Stories': 1}


## 5 · Which objects can be keys?
Keys must be **hashable**: their hash must stay stable while used as keys. Strings and integers work. Lists do not.

Use `counts.get(title, 0)` to avoid a `KeyError` when a title is new. A dictionary stores one value per key; assigning the same key again replaces its old value.

## 7 · Frequency counts: restate → example → plan
**Problem:** Given returned book titles, count how often each title appears.

**Example:** `['Python', 'SQL', 'Python']` → `{'Python': 2, 'SQL': 1}`.

**Plan:** Start with an empty dictionary. For every title, read its old count (or zero), add one, and store it.

**Prediction:** What should an empty list produce? What happens when all titles are identical?

In [18]:
returns = ["Python", "SQL", "Python", "Git", "SQL", "Python"]
counts = {}
for title in returns:
    counts[title] = counts.get(title, 0) + 1
    print(f"After {title!r}: {counts}")
# Final: {'Python': 3, 'SQL': 2, 'Git': 1}

After 'Python': {'Python': 1}
After 'SQL': {'Python': 1, 'SQL': 1}
After 'Python': {'Python': 2, 'SQL': 1}
After 'Git': {'Python': 2, 'SQL': 1, 'Git': 1}
After 'SQL': {'Python': 2, 'SQL': 2, 'Git': 1}
After 'Python': {'Python': 3, 'SQL': 2, 'Git': 1}


## 9 · Turn the plan into a reusable function
Keep the input and output clear. This function counts exact strings: `Python` and `python` are different keys. Decide whether case and extra spaces should matter before changing the data.

**Explain aloud:** Why is the default zero rather than one?

In [19]:
def count_titles(titles):
    """Return a dictionary mapping each exact title to its frequency."""
    frequencies = {}
    for title in titles:
        frequencies[title] = frequencies.get(title, 0) + 1
    return frequencies

assert count_titles([]) == {}
assert count_titles(["Git"]) == {"Git": 1}
assert count_titles(["Git", "Git"]) == {"Git": 2}
assert count_titles(["Git", "git"]) == {"Git": 1, "git": 1}
print(count_titles(returns))

{'Python': 3, 'SQL': 2, 'Git': 1}


## 11 · The time–memory trade-off
Let **n** be the number of titles and **k** the number of distinct titles.

| Approach | Time | Extra space | Trade-off |
|---|---|---|---|
| Run `titles.count(title)` for every input title | O(n²) | O(k) for result | Easy to express, repeatedly scans the list |
| Update a dictionary in one pass | O(n) average | O(k) | Faster scaling, stores counts by key |

The dictionary does **not** make memory free. Compared with a simple scan for just one chosen title, storing counts for every distinct title uses more memory. For full frequency tables, both approaches above store a result dictionary.

These estimates treat hashing/comparing a title as constant cost; very long strings add their own costs.

In [20]:
def count_titles_repeated_scan(titles):
    return {title: titles.count(title) for title in titles}

assert count_titles_repeated_scan(returns) == count_titles(returns)
for n in [10, 100, 1000]:
    print(f"n={n}: repeated-scan element visits ≈ {n*n}; one-pass updates = {n}")
# This compares work counts, not measured execution times.

n=10: repeated-scan element visits ≈ 100; one-pass updates = 10
n=100: repeated-scan element visits ≈ 10000; one-pass updates = 100
n=1000: repeated-scan element visits ≈ 1000000; one-pass updates = 1000


## 13 · Practice: count genres
Before running the next cell, write your own dictionary-counting loop for:

`['story', 'science', 'story', 'history', 'science']`

**Expected:** `{'story': 2, 'science': 2, 'history': 1}`.

**Extension:** Find genres that appear more than once. Explain why a set alone cannot retain the number of appearances. The next cell is a reference solution; try first.

In [ ]:
genres = ["story", "science", "story", "history", "science"]
genre_counts = count_titles(genres)
repeated_genres = [genre for genre, count in genre_counts.items() if count > 1]
assert genre_counts == {"story": 2, "science": 2, "history": 1}
print(genre_counts)
print("Repeated genres:", repeated_genres)          

{'story': 2, 'science': 2, 'history': 1}
Repeated genres: ['story', 'science']


## 15 · Two-sum: find a pair
**Problem:** Given numbers and a target, return the **indices** of two different items whose sum equals the target. Return `None` if no pair exists. Any valid pair is acceptable.

For `[2, 7, 11, 15]` and target `9`, the answer is `(0, 1)` because `2 + 7 = 9`.

Indices are positions starting at zero. Values are the numbers stored there. One item cannot be used twice, but two separate items may have the same value.

In [9]:
def two_sum_slow(numbers, target):
    """Check all pairs of distinct positions."""
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] + numbers[j] == target:
                return (i, j)
    return None

print(two_sum_slow([2, 7, 11, 15], 9))  # (0, 1)
print(two_sum_slow([3], 6))  # None
# O(n²) worst-case time; O(1) extra space.

(0, 1)
None


## 17 · The complement trick
If the current number is `x`, its required partner is `target - x`, called its **complement**.

Keep a dictionary of previously seen **number → index** pairs.
1. Compute the complement.
2. If it is already in the dictionary, return the earlier index and current index.
3. Otherwise, store the current number and index.

**Important:** Check *before* storing. Otherwise `[3]` with target `6` could incorrectly reuse the same position.

In [22]:
def two_sum(numbers, target):
    """Return indices of a matching pair, or None; do not reuse an item."""
    seen = {}
    for index, number in enumerate(numbers):
        complement = target - number
        if complement in seen:
            return (seen[complement], index)
        seen[number] = index
    return None

print(two_sum([2, 7, 11, 15], 9))  # (0, 1)
print(two_sum([3, 3], 6))  # (0, 1): two separate items

(0, 1)
(0, 1)


## 19 · Trace it by hand
For `[4, 1, 6]`, target `7`:

| Index | Number | Needed | Dictionary before check | Action |
|---|---|---|---|---|
| 0 | 4 | 3 | `{}` | Store `4: 0` |
| 1 | 1 | 6 | `{4: 0}` | Store `1: 1` |
| 2 | 6 | 1 | `{4: 0, 1: 1}` | Found; return `(1, 2)` |

**Trade-off:** O(n) average time and O(n) extra space, versus the nested-loop method's O(n²) time and O(1) extra space. Hashing is especially useful when the input is unsorted and original indices matter.

In [11]:
assert two_sum([], 5) is None
assert two_sum([3], 6) is None
assert two_sum([3, 3], 6) == (0, 1)
assert two_sum([1, 2, 3], 20) is None
assert two_sum([-3, 7, 2], 4) == (0, 1)
assert two_sum([0, 0], 0) == (0, 1)
assert two_sum([4, 1, 6], 7) == (1, 2)
print("All two-sum checks passed.")

All two-sum checks passed.


## 21 · Break it on purpose
The next function deliberately stores the current item **too early**. Predict the result for `[3]`, target `6`.

After observing the bug, explain why the correct function in cell 18 avoids it. This is a failure experiment, not a recommended implementation.

In [12]:
def two_sum_broken(numbers, target):
    seen = {}
    for index, number in enumerate(numbers):
        seen[number] = index  # BUG: store before checking
        if target - number in seen:
            return (seen[target - number], index)
    return None

print("Broken:", two_sum_broken([3], 6))  # (0, 0): invalid reuse
print("Correct:", two_sum([3], 6))  # None
assert two_sum_broken([3], 6) == (0, 0)

Broken: (0, 0)
Correct: None


## 23 · Class question round: questions 1–10
Ask learners first; use the brief answers as presenter notes.

1. What is a dictionary? **Key–value storage.**
2. In a frequency dictionary, what is the key? **The item being counted.**
3. What is the value? **Its frequency.**
4. What does `get(key, 0)` do for a missing key? **Returns zero without inserting it.**
5. Why add one after each item? **One more occurrence has been seen.**
6. What is the result for an empty list? **An empty dictionary.**
7. Can a list be a dictionary key? **No; it is unhashable.**
8. Does hashing guarantee O(1) lookup? **No; O(1) is an average-case estimate.**
9. How much space does a frequency dictionary use? **O(k), for k distinct items.**
10. Why not call `count()` for every item? **Repeated scans take O(n²) time; a one-pass dictionary uses O(n) average time but retains a table.**

## 24 · Class question round: questions 11–20
11. What does two-sum return here? **Two distinct indices, or None.**
12. What is the complement of 4 for target 10? **6.**
13. What does `seen` store? **Previously visited number → index.**
14. Why check before storing? **To avoid reusing the current item.**
15. Can `[3, 3]` make 6? **Yes, at indices 0 and 1.**
16. Can `[3]` make 6? **No; there is only one item.**
17. Do negative numbers work? **Yes; complement arithmetic still applies.**
18. What is the fast method's trade-off? **O(n) average time using O(n) extra space.**
19. When might nested loops be reasonable? **Tiny inputs or tight memory limits; slower scaling but constant extra space.**
20. Why is a set insufficient when returning indices? **It records membership, not the earlier index; a dictionary stores both.**

**Presenter prompt:** Ask learners to explain *why*, not just recite complexity labels. Have one learner trace a new example without looking at the solution.

## 25 · Friday task: simulate a returns desk
The source plan assigns Member 6 a **queue simulation**, separate from Wednesday's hashing topic.

A queue is **FIFO**: first in, first out. Use `collections.deque`: removing the leftmost item with `popleft()` is O(1), whereas `list.pop(0)` shifts remaining items and costs O(n).

**Model assumptions:** one desk, arrival order already sorted, minutes from opening, no breaks, no priority customers, fixed service durations per customer. For equal arrival times, retain input order.

`start = max(arrival, previous finish)`  
`wait = start - arrival`  
`finish = start + service duration`

Wait is time *before service starts*, not total time at the desk.

In [13]:
from collections import deque

def simulate_returns(customers):
    """Simulate (name, arrival_minute, service_minutes), in arrival order."""
    queue = deque(customers)
    desk_free = 0
    records = []
    while queue:
        name, arrival, service = queue.popleft()
        start = max(arrival, desk_free)
        finish = start + service
        records.append({"name": name, "arrival": arrival, "start": start,
                        "wait": start - arrival, "finish": finish})
        desk_free = finish
    return records

customers = [("Ada", 0, 3), ("Bayo", 1, 2), ("Chi", 5, 1)]
records = simulate_returns(customers)
for row in records:
    print(row)
# Waits: Ada 0, Bayo 2, Chi 0 minutes.

{'name': 'Ada', 'arrival': 0, 'start': 0, 'wait': 0, 'finish': 3}
{'name': 'Bayo', 'arrival': 1, 'start': 3, 'wait': 2, 'finish': 5}
{'name': 'Chi', 'arrival': 5, 'start': 5, 'wait': 0, 'finish': 6}


## 27 · Measure, test, and explain
Here, “measure wait times” means simulated waiting minutes, not the program's execution time.

For the example, average wait is `(0 + 2 + 0) / 3 ≈ 0.67` minutes. Maximum wait is 2 minutes.

**Limits:** The function assumes valid, sorted arrival times and nonnegative service times; it does not validate them. It models a known arrival list, not a live service. It uses O(n) time and O(n) space including its queue and output. A direct loop would also work; `deque` explicitly demonstrates FIFO processing.

In [14]:
waits = [record["wait"] for record in records]
average_wait = sum(waits) / len(waits) if waits else 0
maximum_wait = max(waits, default=0)
print(f"Average wait: {average_wait:.2f} minutes")
print(f"Maximum wait: {maximum_wait} minutes")
assert waits == [0, 2, 0]
assert simulate_returns([]) == []
assert simulate_returns([("Dele", 10, 2)])[0]["wait"] == 0
assert [r["wait"] for r in simulate_returns([("A", 0, 2), ("B", 0, 1)])] == [0, 2]
print("Queue checks passed.")

Average wait: 0.67 minutes
Maximum wait: 2 minutes
Queue checks passed.


## 29 · Independent practice and Friday hand-off
**Try without copying the reference functions:**
1. Count `['A', 'B', 'A', 'C', 'B', 'A']`. Expected: A=3, B=2, C=1.
2. Solve `[5, 2, 8, 4]`, target 10. Expected indices: `(1, 2)`.
3. Explain why target 10 with `[5]` has no answer.
4. Simulate `[("A", 0, 4), ("B", 1, 2), ("C", 2, 1)]`. Expected waits: 0, 3, 4; average: 7/3 minutes.
5. Extension: reject unsorted arrivals or negative service durations with a clear `ValueError`.

**Friday deliverable:** queue function, fictional example, per-customer waits, average/max wait, edge-case tests, and a short limitations note. Explain the logic to a reviewer before merge.

**AI disclosure:** This lesson was AI-assisted. Follow the repository rule: hand-write the core logic first; record AI help and how you checked it. Running notebook examples is not a substitute for explaining them.

## 30 · Recap and exit ticket
- A **frequency dictionary** stores item → count.
- A **two-sum dictionary** stores number → earlier index.
- Hashing often exchanges **extra memory for fewer repeated searches**.
- Check a two-sum complement **before** inserting the current item.
- A returns-desk queue is FIFO; wait = service start − arrival.

**Exit ticket:** Explain the time–memory trade-off in your own words, trace `[2, 5, 2]` with target `4`, and give one input that catches the self-reuse bug.

**Presenter answer:** Two-sum returns `(0, 2)`; `[3]` with target `6` catches self-reuse. Hashing stores earlier values so it can avoid checking every possible pair.

**Reflect:** What failed? What test caught it? When would you choose a simpler but slower approach?